# Chess Dataset — Data Acquisition & Parsing
### CMPS344 Applied Data Science — Phase 2
---
**Sources:**
- `data.pgn` — 50,000 chess games in standard PGN notation
- `data_uci.pgn` — Same games with moves in UCI coordinate notation
- `stockfish.csv` — Per-game Stockfish centipawn evaluations

**Goal:** Parse all three files, extract features, merge into a single clean DataFrame, and save as `games.csv`.

## 1. Imports & Configuration

In [1]:
# uncomment this line to install dependencies (first run only)
# %pip install python-chess requests

In [2]:
import re
import io
import requests
import pandas as pd
import numpy as np
import chess
import chess.pgn
from collections import Counter

# ── Paths (update to match your folder structure) ────────────────────────────
PGN_PATH              = "../data/data.pgn"
UCI_PATH              = "../data/data_uci.pgn"
SF_PATH               = "../data/stockfish.csv"
LICHESS_PATH          = "../data/chess_games.csv"
LICHESS_SF_PATH       = "../data/lichess_stockfish.csv"
OUTPUT_PATH           = "../data/games.csv"
OUTPUT_MERGED_PATH    = "../data/merged_games.csv"
ECO_PATH              = "../data/eco_openings.csv"

print("Libraries loaded successfully.")

Libraries loaded successfully.


## 2. Download ECO Opening Database (Third Data Source)
Downloads the official Lichess opening database (public domain) from GitHub.
Files `a.tsv` through `e.tsv` cover all ECO codes A00–E99.

This is our **third data source** — merged with the PGN and Stockfish data to add
opening names (e.g. "Sicilian Defense: Najdorf Variation") and ECO codes per game.

> **Citation:** lichess-org/chess-openings, https://github.com/lichess-org/chess-openings (public domain)

In [3]:
def download_eco_database(save_path: str = "eco_openings.csv") -> pd.DataFrame:
    """
    Download the Lichess ECO opening database from GitHub and save locally.
    Covers all ECO codes A00-E99 across 5 TSV files (a.tsv through e.tsv).

    Source: https://github.com/lichess-org/chess-openings (public domain)
    Citation: lichess-org/chess-openings, github.com/lichess-org/chess-openings
    """
    url_template = "https://raw.githubusercontent.com/lichess-org/chess-openings/master/{}.tsv"
    frames = []

    for letter in list("abcde"):
        url = url_template.format(letter)
        try:
            resp = requests.get(url, timeout=20)
            resp.raise_for_status()

            # ── DEBUG: print raw content so we can see what we're getting ────
            raw = resp.text
            print(f"  {letter}.tsv — HTTP {resp.status_code}, {len(raw)} chars")
            print(f"  First line: {repr(raw.splitlines()[0])}")
            print(f"  Second line: {repr(raw.splitlines()[1]) if len(raw.splitlines()) > 1 else 'N/A'}")

            # ── Auto-detect columns from header row ──────────────────────────
            df_letter = pd.read_csv(io.StringIO(raw), sep="\t")
            print(f"  Columns detected: {list(df_letter.columns)}")
            print(f"  Rows: {len(df_letter)}")

            df_letter["eco_family"] = letter.upper()
            frames.append(df_letter)
            print(f"  ✓ Appended successfully\n")

        except Exception as e:
            print(f"  ✗ {letter}.tsv FAILED: {type(e).__name__}: {e}\n")

    if not frames:
        raise RuntimeError(
            "All ECO downloads failed — check the debug output above for clues.\n"
            "Common fixes:\n"
            "  - If HTTP 200 but 0 rows: the TSV format changed, check the column names printed above\n"
            "  - If connection error: check your internet / firewall\n"
            "  - If HTTP 403/404: GitHub URL changed, check github.com/lichess-org/chess-openings"
        )

    df_eco = pd.concat(frames, ignore_index=True)

    # ── Standardise column names regardless of what GitHub returns ───────────
    # Expected columns: eco, name, pgn (or moves), uci, epd
    col_map = {}
    for col in df_eco.columns:
        col_lower = col.lower().strip()
        if col_lower == "eco":        col_map[col] = "eco"
        elif col_lower == "name":     col_map[col] = "name"
        elif col_lower in ("pgn", "moves", "san"): col_map[col] = "pgn"
        elif col_lower == "uci":      col_map[col] = "uci"
        elif col_lower == "epd":      col_map[col] = "epd"
    df_eco = df_eco.rename(columns=col_map)

    if "pgn" not in df_eco.columns:
        raise RuntimeError(
            f"Could not find a moves/pgn column. Columns found: {list(df_eco.columns)}\n"
            "Update the col_map dict above to match."
        )

    # ── Normalise move sequences for prefix matching ──────────────────────────
    def normalise_moves(pgn_str: str) -> str:
        if not isinstance(pgn_str, str):
            return ""
        cleaned = re.sub(r"\d+\.\s*", "", pgn_str)
        return " ".join(cleaned.split()).strip()

    df_eco["moves_normalised"] = df_eco["pgn"].apply(normalise_moves)
    df_eco.to_csv(save_path, index=False)

    print(f"Total ECO entries : {len(df_eco):,}")
    print(f"ECO family counts :\n{df_eco['eco_family'].value_counts().sort_index().to_string()}")
    print(f"Saved to '{save_path}'")
    return df_eco


# ── Load from disk if already downloaded, otherwise fetch ────────────────────
import os
if os.path.exists(ECO_PATH):
    print(f"Found cached '{ECO_PATH}', loading from disk...")
    df_eco = pd.read_csv(ECO_PATH)
    df_eco["moves_normalised"] = df_eco["moves_normalised"].fillna("")
    print(f"Loaded {len(df_eco):,} ECO entries.")
else:
    df_eco = download_eco_database(ECO_PATH)

df_eco.head(5)

Found cached '../data/eco_openings.csv', loading from disk...
Loaded 3,641 ECO entries.


,eco,name,pgn,eco_family,moves_normalised
0,A00,Amar Opening,1. Nh3,A,Nh3
1,A00,Amar Opening: Paris Gambit,1. Nh3 d5 2. g3 e5 3. f4,A,Nh3 d5 g3 e5 f4
2,A00,"Amar Opening: Paris Gambit, Gent Gambit",1. Nh3 d5 2. g3 e5 3. f4 Bxh3 4. Bxh3 exf4 5. ...,A,Nh3 d5 g3 e5 f4 Bxh3 Bxh3 exf4 O-O fxg3 hxg3
3,A00,Amsterdam Attack,1. e3 e5 2. c4 d6 3. Nc3 Nc6 4. b3 Nf6,A,e3 e5 c4 d6 Nc3 Nc6 b3 Nf6
4,A00,Anderssen's Opening,1. a3,A,a3


## 3. Parse `data.pgn` with `python-chess`
Uses the `python-chess` library instead of regex for robust, standards-compliant parsing.

**Extra features now extracted vs the old regex parser:**

| Feature | Description |
|---|---|
| `white_castled` | Did White castle? (bool) |
| `black_castled` | Did Black castle? (bool) |
| `white_castle_side` | 'kingside', 'queenside', or 'none' |
| `black_castle_side` | 'kingside', 'queenside', or 'none' |
| `termination` | 'checkmate', 'resignation', 'draw', or 'unknown' |
| `num_captures` | Total captures in the game |

In [4]:
def parse_pgn(filepath: str) -> pd.DataFrame:
    """
    Parse a PGN file using python-chess and return a DataFrame with one row per game.
    
    Extracted columns:
        event_id         : int   — game identifier
        white_elo        : int   — White player Elo
        black_elo        : int   — Black player Elo
        result           : str   — '1-0', '0-1', or '1/2-1/2'
        num_moves        : int   — number of full moves played
        white_castled    : bool  — whether White castled
        black_castled    : bool  — whether Black castled
        white_castle_side: str   — 'kingside', 'queenside', or 'none'
        black_castle_side: str   — 'kingside', 'queenside', or 'none'
        num_captures     : int   — total number of captures
        termination      : str   — how the game ended
        moves_san        : str   — full move sequence in SAN (for ECO matching)
    """
    records = []

    with open(filepath, "r", encoding="utf-8") as f:
        while True:
            game = chess.pgn.read_game(f)
            if game is None:
                break

            headers = game.headers

            # ── Skip games missing essential fields ──────────────────────────
            try:
                white_elo = int(headers.get("WhiteElo", ""))
                black_elo = int(headers.get("BlackElo", ""))
            except ValueError:
                continue

            result    = headers.get("Result", "*")
            event_id  = int(headers.get("Event", 0))

            # ── Walk through moves to extract board-level features ────────────
            board = game.board()
            moves_san       = []
            num_captures    = 0
            white_castled   = False
            black_castled   = False
            white_castle_side = "none"
            black_castle_side = "none"

            for move in game.mainline_moves():
                san = board.san(move)
                moves_san.append(san)

                # Detect captures
                if board.is_capture(move):
                    num_captures += 1

                # Detect castling
                if board.is_castling(move):
                    is_kingside = board.is_kingside_castling(move)
                    side = "kingside" if is_kingside else "queenside"
                    if board.turn == chess.WHITE:
                        white_castled     = True
                        white_castle_side = side
                    else:
                        black_castled     = True
                        black_castle_side = side

                board.push(move)

            num_moves  = len(moves_san) // 2  # full moves
            moves_text = " ".join(moves_san)

            # ── Infer termination ─────────────────────────────────────────────
            if board.is_checkmate():
                termination = "checkmate"
            elif result in ("1/2-1/2",):
                termination = "draw"
            elif result in ("1-0", "0-1"):
                termination = "resignation"
            else:
                termination = "unknown"

            records.append({
                "event_id":          event_id,
                "white_elo":         white_elo,
                "black_elo":         black_elo,
                "result":            result,
                "num_moves":         num_moves,
                "white_castled":     white_castled,
                "black_castled":     black_castled,
                "white_castle_side": white_castle_side,
                "black_castle_side": black_castle_side,
                "num_captures":      num_captures,
                "termination":       termination,
                "moves_san":         moves_text,
            })

    df = pd.DataFrame(records)
    print(f"Parsed {len(df):,} games from {filepath}")
    print(f"\nResult distribution:\n{df['result'].value_counts().to_string()}")
    print(f"\nTermination distribution:\n{df['termination'].value_counts().to_string()}")
    return df


df_pgn = parse_pgn(PGN_PATH)
df_pgn.head(3)

Parsed 25,000 games from ../data/data.pgn

Result distribution:
result
1-0        9704
1/2-1/2    7791
0-1        7505

Termination distribution:
termination
resignation    16653
draw            7790
checkmate        557


,event_id,white_elo,black_elo,result,num_moves,white_castled,black_castled,white_castle_side,black_castle_side,num_captures,termination,moves_san
0,1,2354,2411,1/2-1/2,19,True,True,kingside,kingside,9,draw,Nf3 Nf6 c4 c5 b3 g6 Bb2 Bg7 e3 O-O Be2 b6 O-O ...
1,2,2523,2460,1/2-1/2,6,True,False,kingside,none,2,draw,e4 e5 Nf3 Nf6 d4 Nxe4 Nxe5 d6 Nf3 d5 Bd3 Nd6 O-O
2,3,1915,1999,0-1,53,True,True,kingside,kingside,23,resignation,e4 d5 exd5 Nf6 d4 Nxd5 Nf3 g6 Be2 Bg7 c4 Nb6 N...


## 4. Match Games to ECO Openings
For each game, try to match the opening moves against the ECO database using
a **longest-prefix match** — the more moves played in the opening, the more
specific the ECO code we can assign.

In [5]:
def build_eco_lookup(df_eco: pd.DataFrame) -> dict:
    """
    Build a prefix lookup dictionary from the ECO database.
    Maps normalised move prefix → (eco_code, opening_name, eco_family).
    
    Parameters
    ----------
    df_eco : DataFrame from download_eco_database()
    
    Returns
    -------
    dict keyed by normalised move string
    """
    lookup = {}
    for _, row in df_eco.iterrows():
        key = row["moves_normalised"].strip()
        if key:
            lookup[key] = (row["eco"], row["name"], row["eco_family"])
    return lookup


def match_eco(moves_san: str, lookup: dict, max_depth: int = 10) -> tuple:
    """
    Match a game's moves against the ECO lookup using longest-prefix matching.
    
    Tries progressively shorter prefixes until a match is found.
    
    Parameters
    ----------
    moves_san : str   — full SAN move sequence from parse_pgn()
    lookup    : dict  — from build_eco_lookup()
    max_depth : int   — maximum number of half-moves to try
    
    Returns
    -------
    (eco_code, opening_name, eco_family) or ('Unknown', 'Unknown', 'Unknown')
    """
    if not isinstance(moves_san, str):
        return ("Unknown", "Unknown", "Unknown")

    # Remove annotations like '!', '?', '+', '#'
    clean = re.sub(r"[+#!?]", "", moves_san).strip()
    tokens = clean.split()

    # Try longest prefix first, shrink until a match is found
    for depth in range(min(max_depth, len(tokens)), 0, -1):
        prefix = " ".join(tokens[:depth])
        if prefix in lookup:
            return lookup[prefix]

    return ("Unknown", "Unknown", "Unknown")


# ── Build lookup and apply to all games ──────────────────────────────────────
print("Building ECO lookup table...")
eco_lookup = build_eco_lookup(df_eco)
print(f"Lookup table size: {len(eco_lookup):,} entries")

print("\nMatching games to ECO codes (this may take a minute)...")
eco_results = df_pgn["moves_san"].apply(lambda m: match_eco(m, eco_lookup))

df_pgn["eco_code"]     = eco_results.apply(lambda x: x[0])
df_pgn["opening_name"] = eco_results.apply(lambda x: x[1])
df_pgn["eco_family"]   = eco_results.apply(lambda x: x[2])

matched = (df_pgn["eco_code"] != "Unknown").sum()
print(f"\nGames matched to ECO opening: {matched:,} / {len(df_pgn):,} ({matched/len(df_pgn)*100:.1f}%)")
print(f"\nTop 10 openings:")
print(df_pgn["opening_name"].value_counts().head(10).to_string())

Building ECO lookup table...
Lookup table size: 3,641 entries

Matching games to ECO codes (this may take a minute)...

Games matched to ECO opening: 25,000 / 25,000 (100.0%)

Top 10 openings:
opening_name
Zukertort Opening                         1063
Sicilian Defense: Najdorf Variation        680
Pirc Defense                               394
Horwitz Defense                            368
Indian Defense: Knights Variation          357
Zukertort Opening: Sicilian Invitation     294
Sicilian Defense: Modern Variations        289
Ruy Lopez: Closed                          283
Caro-Kann Defense                          282
Indian Defense: Anti-Nimzo-Indian          279


## 5. Parse `data_uci.pgn`
Extracts move sequences in UCI coordinate format (e.g. `e2e4 e7e5`).
We keep this as a separate feature — UCI notation is useful for pattern matching and
is a distinct representation from the SAN moves already parsed from `data.pgn`.

In [6]:
def parse_uci(filepath: str) -> pd.DataFrame:
    """
    Parse a UCI-format PGN file and return a DataFrame with one row per game.

    Extracted columns:
        event_id  : int — game identifier (links back to data.pgn)
        moves_uci : str — full move sequence in UCI coordinate notation
                          e.g. 'e2e4 e7e5 g1f3 b8c6'
    """
    with open(filepath, "r", encoding="utf-8") as f:
        content = f.read()

    game_blocks = re.split(r"\n(?=\[Event )", content.strip())
    records = []

    for block in game_blocks:
        event_match = re.search(r'\[Event "([^"]+)"\]', block)
        if not event_match:
            continue

        move_lines = [
            line for line in block.split("\n")
            if line.strip() and not line.startswith("[")
        ]
        moves_raw  = " ".join(move_lines)
        moves_clean = re.sub(
            r"\s*(1-0|0-1|1/2-1/2|\*)\s*$", "", moves_raw
        ).strip()

        records.append({
            "event_id":  int(event_match.group(1)),
            "moves_uci": moves_clean,
        })

    df = pd.DataFrame(records)
    print(f"Parsed {len(df):,} games from {filepath}")
    return df


df_uci = parse_uci(UCI_PATH)
df_uci.head(3)

Parsed 50,000 games from ../data/data_uci.pgn


,event_id,moves_uci
0,1,g1f3 g8f6 c2c4 c7c5 b2b3 g7g6 c1b2 f8g7 e2e3 e...
1,2,e2e4 e7e5 g1f3 g8f6 d2d4 f6e4 f3e5 d7d6 e5f3 d...
2,3,e2e4 d7d5 e4d5 g8f6 d2d4 f6d5 g1f3 g7g6 f1e2 f...


## 6. Parse `stockfish.csv` & Extract Evaluation Features
Each row contains space-separated centipawn scores for every half-move in the game.
- **Positive score** = advantage for White
- **Negative score** = advantage for Black

We compute the following features per game:

| Feature | Description |
|---|---|
| `white_acl` | Average centipawn loss per move for White |
| `black_acl` | Average centipawn loss per move for Black |
| `white_blunders` | Moves where White lost ≥ 100 centipawns |
| `black_blunders` | Moves where Black lost ≥ 100 centipawns |
| `white_mistakes` | Moves where White lost 50–99 centipawns |
| `black_mistakes` | Moves where Black lost 50–99 centipawns |
| `final_eval` | Centipawn evaluation at game's last move |
| `max_white_advantage` | Peak advantage White held during the game |
| `max_black_advantage` | Peak advantage Black held (most negative value) |
| `game_sharpness` | Std deviation of all scores — higher = more tactical |

In [7]:
def extract_stockfish_features(filepath: str) -> pd.DataFrame:
    """
    Parse stockfish.csv and compute per-game evaluation features.
    
    Parameters
    ----------
    filepath : str — path to stockfish.csv
    
    Returns
    -------
    pd.DataFrame with one row per game and derived evaluation features
    """
    df_sf = pd.read_csv(filepath)
    records = []

    for _, row in df_sf.iterrows():
        try:
            scores = list(map(int, str(row["MoveScores"]).split()))
        except (ValueError, AttributeError):
            continue

        if len(scores) < 2:
            continue

        # ── Split scores by player ───────────────────────────────────────────
        # After White's move: even indices (0, 2, 4, ...)
        # After Black's move: odd indices  (1, 3, 5, ...)
        white_scores = scores[0::2]   # evaluation after White moves
        black_scores = scores[1::2]   # evaluation after Black moves

        # ── Helper: average centipawn loss ───────────────────────────────────
        def avg_centipawn_loss(scores_list, is_white: bool) -> float:
            losses = []
            for i in range(1, len(scores_list)):
                prev, curr = scores_list[i - 1], scores_list[i]
                # White wants score to go up; Black wants it to go down
                loss = (prev - curr) if is_white else (curr - prev)
                if loss > 0:
                    losses.append(loss)
            return round(np.mean(losses), 2) if losses else 0.0

        # ── Helper: count errors above a centipawn threshold ─────────────────
        def count_errors(scores_list, is_white: bool, threshold: int) -> int:
            count = 0
            for i in range(1, len(scores_list)):
                prev, curr = scores_list[i - 1], scores_list[i]
                loss = (prev - curr) if is_white else (curr - prev)
                if loss >= threshold:
                    count += 1
            return count

        records.append({
            "event_id":            row["Event"],
            "total_half_moves":    len(scores),
            "white_acl":           avg_centipawn_loss(white_scores, is_white=True),
            "black_acl":           avg_centipawn_loss(black_scores, is_white=False),
            "white_blunders":      count_errors(white_scores, True,  threshold=100),
            "black_blunders":      count_errors(black_scores, False, threshold=100),
            "white_mistakes":      count_errors(white_scores, True,  threshold=50) -
                                   count_errors(white_scores, True,  threshold=100),
            "black_mistakes":      count_errors(black_scores, False, threshold=50) -
                                   count_errors(black_scores, False, threshold=100),
            "final_eval":          scores[-1],
            "max_white_advantage": max(scores),
            "max_black_advantage": min(scores),
            "game_sharpness":      round(float(np.std(scores)), 2),
        })

    df = pd.DataFrame(records)
    print(f"Extracted Stockfish features for {len(df):,} games")
    print(df[["white_acl", "black_acl", "white_blunders", "black_blunders"]].describe().round(2).to_string())
    return df


df_sf = extract_stockfish_features(SF_PATH)
df_sf.head(3)

Extracted Stockfish features for 47,174 games
       white_acl  black_acl  white_blunders  black_blunders
count   47174.00   47174.00        47174.00        47174.00
mean       57.14      63.49            1.44            1.61
std       100.39     107.61            1.97            2.06
min         0.00       0.00            0.00            0.00
25%        16.90      17.52            0.00            0.00
50%        25.73      28.62            1.00            1.00
75%        48.11      56.15            2.00            3.00
max      2111.92    1880.20           23.00           21.00


,event_id,total_half_moves,white_acl,black_acl,white_blunders,black_blunders,white_mistakes,black_mistakes,final_eval,max_white_advantage,max_black_advantage,game_sharpness
0,1,38,13.67,15.11,0,0,0,0,54,73,-26,26.43
1,2,13,12.00,12.33,0,0,0,0,55,55,14,11.54
2,3,106,312.33,203.29,7,2,0,0,-11544,93,-11544,2286.13


## 7. Merge All Sources into One DataFrame
Join on `event_id` (the shared game identifier across all three files).

In [8]:
def merge_datasets(
    df_pgn: pd.DataFrame,
    df_uci: pd.DataFrame,
    df_sf: pd.DataFrame
) -> pd.DataFrame:
    """
    Merge the three parsed DataFrames on event_id using inner joins.
    
    Sources:
        df_pgn : game metadata + ECO opening info  (from data.pgn + lichess ECO)
        df_uci : UCI move sequences                 (from data_uci.pgn)
        df_sf  : Stockfish evaluation features      (from stockfish.csv)
    
    Returns
    -------
    Merged DataFrame with all features from all three sources
    """
    # Keep only moves_uci from UCI file (metadata already in df_pgn)
    df_uci_slim = df_uci[["event_id", "moves_uci"]]

    df = (
        df_pgn
        .merge(df_uci_slim, on="event_id", how="inner")
        .merge(df_sf,       on="event_id", how="inner")
    )

    print(f"Merged DataFrame shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    return df


df = merge_datasets(df_pgn, df_uci, df_sf)
df.head(3)

Merged DataFrame shape: (23636, 27)
Columns: ['event_id', 'white_elo', 'black_elo', 'result', 'num_moves', 'white_castled', 'black_castled', 'white_castle_side', 'black_castle_side', 'num_captures', 'termination', 'moves_san', 'eco_code', 'opening_name', 'eco_family', 'moves_uci', 'total_half_moves', 'white_acl', 'black_acl', 'white_blunders', 'black_blunders', 'white_mistakes', 'black_mistakes', 'final_eval', 'max_white_advantage', 'max_black_advantage', 'game_sharpness']


,event_id,white_elo,black_elo,result,num_moves,white_castled,black_castled,white_castle_side,black_castle_side,num_captures,...,white_acl,black_acl,white_blunders,black_blunders,white_mistakes,black_mistakes,final_eval,max_white_advantage,max_black_advantage,game_sharpness
0,1,2354,2411,1/2-1/2,19,True,True,kingside,kingside,9,...,13.67,15.11,0,0,0,0,54,73,-26,26.43
1,2,2523,2460,1/2-1/2,6,True,False,kingside,none,2,...,12.00,12.33,0,0,0,0,55,55,14,11.54
2,3,1915,1999,0-1,53,True,True,kingside,kingside,23,...,312.33,203.29,7,2,0,0,-11544,93,-11544,2286.13


## 8. Feature Engineering
Derive additional features that carry useful signal for both tasks:

| Feature | Formula | Rationale |
|---|---|---|
| `elo_gap` | `white_elo - black_elo` | Relative strength difference |
| `avg_elo` | `(white_elo + black_elo) / 2` | Overall game quality proxy |
| `elo_bucket_white` | Binned white Elo | Target variable for Elo classification task |
| `elo_bucket_black` | Binned black Elo | Target variable for Elo classification task |
| `acl_gap` | `white_acl - black_acl` | Relative accuracy difference |
| `winner` | Encoded result | Target variable for winner prediction task |

In [ ]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add derived features and encode target variables.

    Parameters
    ----------
    df : merged DataFrame from merge_datasets()

    Returns
    -------
    DataFrame with additional engineered columns
    """
    df = df.copy()

    # ── Elo classification target (bin into skill tiers) ─────────────────────
    # white_elo is used ONLY to derive the target, then kept in dataset
    # but must be dropped before any model training (done in preprocessing notebook)
    elo_bins   = [0, 1000, 1500, 2000, 2500, 9999]
    elo_labels = ["Beginner", "Intermediate", "Advanced", "Expert", "Master"]

    df["elo_bucket_white"] = pd.cut(
        df["white_elo"], bins=elo_bins, labels=elo_labels, right=False
    )
    df["elo_bucket_black"] = pd.cut(
        df["black_elo"], bins=elo_bins, labels=elo_labels, right=False
    )

    # ── Ordinal encoding of target ────────────────────────────────────────────
    # Beginner=0, Intermediate=1, Advanced=2, Expert=3, Master=4
    ordinal_map = {"Beginner": 0, "Intermediate": 1, "Advanced": 2,
                   "Expert": 3, "Master": 4}
    df["elo_bucket_white_enc"] = df["elo_bucket_white"].map(ordinal_map)
    df["elo_bucket_black_enc"] = df["elo_bucket_black"].map(ordinal_map)

    # ── Accuracy gap ─────────────────────────────────────────────────────────
    df["acl_gap"] = (df["white_acl"] - df["black_acl"]).round(2)

    # ── Winner target variable ────────────────────────────────────────────────
    result_map_binary     = {"1-0": 1, "0-1": 0}
    result_map_multiclass = {"0-1": 0, "1/2-1/2": 1, "1-0": 2}

    df["winner_binary"]     = df["result"].map(result_map_binary)
    df["winner_multiclass"] = df["result"].map(result_map_multiclass)

    print("Engineered features added.")
    print(f"\nElo bucket distribution (White):\n{df['elo_bucket_white'].value_counts().sort_index().to_string()}")

Engineered features added.

Elo bucket distribution (White):
elo_bucket_white
Expert          15216
Master           4422
Advanced         3829
Intermediate      169
Beginner            0

Winner (multiclass) distribution:
winner_multiclass
2    8913
1    7774
0    6949


,event_id,white_elo,black_elo,result,num_moves,white_castled,black_castled,white_castle_side,black_castle_side,num_captures,...,max_white_advantage,max_black_advantage,game_sharpness,elo_gap,avg_elo,elo_bucket_white,elo_bucket_black,acl_gap,winner_binary,winner_multiclass
0,1,2354,2411,1/2-1/2,19,True,True,kingside,kingside,9,...,73,-26,26.43,-57,2382.0,Expert,Expert,-1.44,NaN,1
1,2,2523,2460,1/2-1/2,6,True,False,kingside,none,2,...,55,14,11.54,63,2492.0,Master,Expert,-0.33,NaN,1
2,3,1915,1999,0-1,53,True,True,kingside,kingside,23,...,93,-11544,2286.13,-84,1957.0,Advanced,Advanced,109.04,0.0,0


## 9. Data Validation Function
Required for Phase 2. Checks for missing values, duplicates, data types, and class distribution.

In [ ]:
def validation_report(df: pd.DataFrame) -> None:
    """
    Print a comprehensive data validation report.

    Covers:
        - Shape (rows, columns)
        - Data types per column
        - Missing values count and percentage
        - Duplicate rows
        - Class distribution for target variable
    """
    print("=" * 60)
    print("DATA VALIDATION REPORT")
    print("=" * 60)

    print(f"\nShape: {df.shape[0]:,} rows × {df.shape[1]} columns")

    print("\n── Data Types ──────────────────────────────────────────────")
    print(df.dtypes.to_string())

    print("\n── Missing Values ──────────────────────────────────────────")
    missing     = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(2)
    missing_df  = pd.DataFrame({"count": missing, "pct": missing_pct})
    missing_df  = missing_df[missing_df["count"] > 0]
    if missing_df.empty:
        print("No missing values found.")
    else:
        print(missing_df.to_string())

    print(f"\n── Duplicates ──────────────────────────────────────────────")
    dup_count = df.duplicated(subset=["event_id"]).sum() if "event_id" in df.columns else "N/A"
    print(f"Duplicate event_id rows: {dup_count}")

    print("\n── Numeric Summary ─────────────────────────────────────────")
    # white_elo included here for context only — not a model input feature
    numeric_cols = [c for c in ["white_elo", "black_elo", "num_moves", "white_acl",
                    "black_acl", "white_blunders", "black_blunders",
                    "acl_gap", "game_sharpness"] if c in df.columns]
    print(df[numeric_cols].describe().round(2).to_string())

    print("\n── Target: elo_bucket_white (categorical) ──────────────────")
    if "elo_bucket_white" in df.columns:
        print(df["elo_bucket_white"].value_counts().sort_index().to_string())

    print("\n── Target: elo_bucket_white_enc (ordinal) ──────────────────")
    if "elo_bucket_white_enc" in df.columns:
        label_map = {0:"Beginner",1:"Intermediate",2:"Advanced",3:"Expert",4:"Master"}
        print(df["elo_bucket_white_enc"].value_counts().sort_index().rename(label_map).to_string())

    print("\n" + "=" * 60)

## 11. Integrate Lichess Dataset
Loads `chess_games.csv` (Lichess casual/club games), harmonises its columns to match
the PGN pipeline schema, then concatenates both datasets into one unified DataFrame.

**Column mapping:**

| Unified column | PGN source | Lichess source |
|---|---|---|
| `white_elo` | `white_elo` | `white_rating` |
| `black_elo` | `black_elo` | `black_rating` |
| `num_moves` | `num_moves` | `turns // 2` |
| `termination` | `termination` | `victory_status` |
| `eco_code` | `eco_code` | `opening_code` |
| `opening_name` | `opening_name` | `opening_fullname` |
| `eco_family` | `eco_family` | `opening_code[0]` |
| `winner_multiclass` | encoded from `result` | encoded from `winner` |
| `has_stockfish` | `True` | `False` |

Stockfish columns (`white_acl`, `black_acl`, blunders, etc.) will be `NaN`
for all Lichess rows — handled during preprocessing via imputation.

In [ ]:
def load_lichess(filepath: str, stockfish_path: str) -> pd.DataFrame:
    """
    Load chess_games.csv (Lichess) and harmonise columns to match the PGN pipeline schema.
    Now also loads pre-computed Stockfish features from lichess_stockfish.csv so that
    all ~43k games in the merged dataset have full engine evaluation features.

    Parameters
    ----------
    filepath       : str — path to chess_games.csv
    stockfish_path : str — path to lichess_stockfish.csv

    Returns
    -------
    DataFrame with unified column names and full Stockfish features,
    ready to concatenate with the PGN pipeline output.
    """
    df_l = pd.read_csv(filepath)
    print(f"Loaded {len(df_l):,} games from '{filepath}'")

    # ── Load and extract Stockfish features ──────────────────────────────────
    print(f"Loading Stockfish features from '{stockfish_path}'...")
    df_sf = extract_stockfish_features(stockfish_path)
    # Rename Event → game_id so we can merge on game_id
    df_sf = df_sf.rename(columns={"event_id": "game_id"})
    print(f"Extracted Stockfish features for {len(df_sf):,} games")

    # ── Merge Lichess games with their Stockfish features ────────────────────
    df_l = df_l.merge(df_sf, on="game_id", how="left")
    missing_sf = df_l["white_acl"].isna().sum()
    if missing_sf > 0:
        print(f"Warning: {missing_sf} games have no Stockfish data (will be NaN)")

    out = pd.DataFrame()

    # ── IDs & source ─────────────────────────────────────────────────────────
    out["event_id"]   = df_l["game_id"]
    out["source"]     = "lichess_csv"

    # ── Elo ───────────────────────────────────────────────────────────────────
    out["white_elo"]  = df_l["white_rating"]
    out["black_elo"]  = df_l["black_rating"]

    # ── Game length ───────────────────────────────────────────────────────────
    out["num_moves"]        = df_l["turns"] // 2
    out["total_half_moves"] = df_l["turns"]

    # ── Termination ───────────────────────────────────────────────────────────
    termination_map = {
        "Resign":      "resignation",
        "Mate":        "checkmate",
        "Out of Time": "timeout",
        "Draw":        "draw",
    }
    out["termination"] = df_l["victory_status"].map(termination_map).fillna("unknown")

    # ── Opening info ──────────────────────────────────────────────────────────
    out["eco_code"]     = df_l["opening_code"]
    out["opening_name"] = df_l["opening_fullname"]
    out["eco_family"]   = df_l["opening_code"].str[0].str.upper()

    # ── Move sequences ────────────────────────────────────────────────────────
    out["moves_san"] = df_l["moves"]
    out["moves_uci"] = np.nan

    # ── Ordinal encoding of target ────────────────────────────────────────────
    # Beginner=0, Intermediate=1, Advanced=2, Expert=3, Master=4
    ordinal_map = {"Beginner": 0, "Intermediate": 1, "Advanced": 2,
                   "Expert": 3, "Master": 4}
    out["elo_bucket_white_categorical"] = out["elo_bucket_white"].map(ordinal_map)
    out["elo_bucket_black_categorical"] = out["elo_bucket_black"].map(ordinal_map)

    # ── Elo classification target ────────────────────────────────────────────
    elo_bins   = [0, 1000, 1500, 2000, 2500, 9999]
    elo_labels = ["Beginner", "Intermediate", "Advanced", "Expert", "Master"]
    out["elo_bucket_white"] = pd.cut(df_l["white_rating"], bins=elo_bins, labels=elo_labels, right=False)
    out["elo_bucket_black"] = pd.cut(df_l["black_rating"], bins=elo_bins, labels=elo_labels, right=False)

    # ── Winner target ─────────────────────────────────────────────────────────
    out["winner_multiclass"] = df_l["winner"].map({"Black": 0, "Draw": 1, "White": 2})
    out["winner_binary"]     = df_l["winner"].map({"White": 1, "Black": 0})

    # ── Stockfish features (now populated from lichess_stockfish.csv) ─────────
    stockfish_cols = [
        "white_acl", "black_acl", "white_blunders", "black_blunders",
        "white_mistakes", "black_mistakes", "final_eval",
        "max_white_advantage", "max_black_advantage", "game_sharpness",
    ]
    for col in stockfish_cols:
        out[col] = df_l[col] if col in df_l.columns else np.nan

    out["acl_gap"] = (out["white_acl"] - out["black_acl"]).round(2)

    # ── Castling — not available in Lichess CSV ───────────────────────────────
    out["white_castled"]     = np.nan
    out["black_castled"]     = np.nan
    out["white_castle_side"] = np.nan
    out["black_castle_side"] = np.nan
    out["num_captures"]      = np.nan

    # ── Source flag ───────────────────────────────────────────────────────────
    out["has_stockfish"] = out["white_acl"].notna()

    print(f"\nLichess harmonised shape: {out.shape}")
    print(f"has_stockfish = True : {out['has_stockfish'].sum():,}")
    print(f"has_stockfish = False: {(~out['has_stockfish']).sum():,}")
    return out


def integrate_datasets(df_pgn_pipeline: pd.DataFrame, lichess_path: str, lichess_sf_path: str) -> pd.DataFrame:
    """
    Concatenate the PGN pipeline output with the Lichess CSV dataset.
    Both sources now have full Stockfish evaluation features.

    Parameters
    ----------
    df_pgn_pipeline : DataFrame — output of engineer_features()
    lichess_path    : str       — path to chess_games.csv
    lichess_sf_path : str       — path to lichess_stockfish.csv

    Returns
    -------
    Unified DataFrame with all ~43k games and full feature coverage
    """
    df_pgn_tagged = df_pgn_pipeline.copy()
    df_pgn_tagged["source"]        = "kaggle_pgn"
    df_pgn_tagged["has_stockfish"] = True

    df_lichess  = load_lichess(lichess_path, lichess_sf_path)
    df_combined = pd.concat([df_pgn_tagged, df_lichess], ignore_index=True, sort=False)
    df_combined["event_id"] = range(1, len(df_combined) + 1)

    print(f"\n── Integration Summary ──────────────────────────────────")
    print(f"Kaggle PGN rows    : {len(df_pgn_tagged):,}")
    print(f"Lichess CSV rows   : {len(df_lichess):,}")
    print(f"Combined total     : {len(df_combined):,}")
    print(f"\nhas_stockfish breakdown:")
    print(df_combined["has_stockfish"].value_counts().to_string())
    print(f"\nelo_bucket_white distribution:")
    print(df_combined["elo_bucket_white"].value_counts().sort_index().to_string())
    print(f"\nMissing white_acl  : {df_combined['white_acl'].isna().sum():,}")
    print(f"Missing black_acl  : {df_combined['black_acl'].isna().sum():,}")
    return df_combined


df_combined = integrate_datasets(df, LICHESS_PATH, LICHESS_SF_PATH)
df_combined.tail(3)

Loaded 20,058 games from '../data/chess_games.csv'
Loading Stockfish features from '../data/lichess_stockfish.csv'...
Extracted Stockfish features for 20,040 games
       white_acl  black_acl  white_blunders  black_blunders
count   20040.00   20040.00        20040.00        20040.00
mean      150.13     102.00            3.13            3.00
std       196.93     111.80            2.53            2.52
min         0.00       0.00            0.00            0.00
25%        48.67      43.77            1.00            1.00
50%        86.96      73.00            3.00            3.00
75%       182.54     128.28            5.00            4.00
max      4000.00    2136.00           20.00           20.00
Extracted Stockfish features for 20,040 games

Lichess harmonised shape: (20058, 35)
has_stockfish = True : 20,040
has_stockfish = False: 18

── Integration Summary ──────────────────────────────────
Kaggle PGN rows    : 23,636
Lichess CSV rows   : 20,058
Combined total     : 43,694

has_stockfi

,event_id,white_elo,black_elo,result,num_moves,white_castled,black_castled,white_castle_side,black_castle_side,num_captures,...,game_sharpness,elo_gap,avg_elo,elo_bucket_white,elo_bucket_black,acl_gap,winner_binary,winner_multiclass,source,has_stockfish
43691,43692,1219,1286,NaN,17,NaN,NaN,NaN,NaN,NaN,...,681.81,-67,1252.0,Intermediate,Intermediate,-115.65,1.0,2,lichess_csv,True
43692,43693,1360,1227,NaN,54,NaN,NaN,NaN,NaN,NaN,...,594.35,133,1294.0,Intermediate,Intermediate,12.68,1.0,2,lichess_csv,True
43693,43694,1235,1339,NaN,39,NaN,NaN,NaN,NaN,NaN,...,638.91,-104,1287.0,Intermediate,Intermediate,263.31,0.0,0,lichess_csv,True


## 12.1. Data Validation Report (Games with stockfish)

In [12]:
validation_report(df)

DATA VALIDATION REPORT

Shape: 23,636 rows × 34 columns

── Data Types ──────────────────────────────────────────────
event_id                  int64
white_elo                 int64
black_elo                 int64
result                   object
num_moves                 int64
white_castled              bool
black_castled              bool
white_castle_side        object
black_castle_side        object
num_captures              int64
termination              object
moves_san                object
eco_code                 object
opening_name             object
eco_family               object
moves_uci                object
total_half_moves          int64
white_acl               float64
black_acl               float64
white_blunders            int64
black_blunders            int64
white_mistakes            int64
black_mistakes            int64
final_eval                int64
max_white_advantage       int64
max_black_advantage       int64
game_sharpness          float64
elo_gap           

## 12.2. Data Validation Report (Games with no stockfish)

In [13]:
validation_report(load_lichess(LICHESS_PATH, LICHESS_SF_PATH))

Loaded 20,058 games from '../data/chess_games.csv'
Loading Stockfish features from '../data/lichess_stockfish.csv'...
Extracted Stockfish features for 20,040 games
       white_acl  black_acl  white_blunders  black_blunders
count   20040.00   20040.00        20040.00        20040.00
mean      150.13     102.00            3.13            3.00
std       196.93     111.80            2.53            2.52
min         0.00       0.00            0.00            0.00
25%        48.67      43.77            1.00            1.00
50%        86.96      73.00            3.00            3.00
75%       182.54     128.28            5.00            4.00
max      4000.00    2136.00           20.00           20.00
Extracted Stockfish features for 20,040 games

Lichess harmonised shape: (20058, 35)
has_stockfish = True : 20,040
has_stockfish = False: 18
DATA VALIDATION REPORT

Shape: 20,058 rows × 35 columns

── Data Types ──────────────────────────────────────────────
event_id                  int64
sourc

## 12.3. Data Validation Report (All games)

In [14]:
validation_report(df_combined)

DATA VALIDATION REPORT

Shape: 43,694 rows × 36 columns

── Data Types ──────────────────────────────────────────────
event_id                  int64
white_elo                 int64
black_elo                 int64
result                   object
num_moves                 int64
white_castled            object
black_castled            object
white_castle_side        object
black_castle_side        object
num_captures            float64
termination              object
moves_san                object
eco_code                 object
opening_name             object
eco_family               object
moves_uci                object
total_half_moves          int64
white_acl               float64
black_acl               float64
white_blunders          float64
black_blunders          float64
white_mistakes          float64
black_mistakes          float64
final_eval              float64
max_white_advantage     float64
max_black_advantage     float64
game_sharpness          float64
elo_gap           

## 13. Save Final Dataset
Save the merged and engineered DataFrame to `merged_games.csv` — this is the input for EDA and modeling.
Save the DataFrame with stockfish only to `games.csv`

In [15]:
def save_dataset(df: pd.DataFrame, output_path: str) -> None:
    """
    Save the final DataFrame to CSV, dropping raw move columns
    that are not needed for modeling (but keeping them available
    in the PGN files if needed later).
    
    Parameters
    ----------
    df          : final engineered DataFrame
    output_path : path to save the CSV file
    """
    # Drop raw move text — not needed in the model CSV
    cols_to_drop = ["moves_pgn", "moves_uci"]
    df_save = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

    df_save.to_csv(output_path, index=False)
    print(f"Saved {len(df_save):,} rows × {df_save.shape[1]} columns to '{output_path}'")
    print(f"\nFinal columns:\n{list(df_save.columns)}")


save_dataset(df_combined, OUTPUT_MERGED_PATH)
save_dataset(df, OUTPUT_PATH)

Saved 43,694 rows × 35 columns to '../data/merged_games.csv'

Final columns:
['event_id', 'white_elo', 'black_elo', 'result', 'num_moves', 'white_castled', 'black_castled', 'white_castle_side', 'black_castle_side', 'num_captures', 'termination', 'moves_san', 'eco_code', 'opening_name', 'eco_family', 'total_half_moves', 'white_acl', 'black_acl', 'white_blunders', 'black_blunders', 'white_mistakes', 'black_mistakes', 'final_eval', 'max_white_advantage', 'max_black_advantage', 'game_sharpness', 'elo_gap', 'avg_elo', 'elo_bucket_white', 'elo_bucket_black', 'acl_gap', 'winner_binary', 'winner_multiclass', 'source', 'has_stockfish']
Saved 23,636 rows × 33 columns to '../data/games.csv'

Final columns:
['event_id', 'white_elo', 'black_elo', 'result', 'num_moves', 'white_castled', 'black_castled', 'white_castle_side', 'black_castle_side', 'num_captures', 'termination', 'moves_san', 'eco_code', 'opening_name', 'eco_family', 'total_half_moves', 'white_acl', 'black_acl', 'white_blunders', 'black_